# Opening Demo: One Scientific Question, Two Workflows

**Question:** How did global near-surface air temperature change relative to 1981–2010?

> Use CDO/NCO to prepare the data.  
> Use Python to understand the data.

This notebook begins with the scientific result, then compares an explicit xarray workflow with a compact CDO/NCO workflow, and finally checks numerical agreement.

In [ ]:
from pathlib import Path
import os

if Path.cwd().name == 'notebooks':
    os.chdir('..')

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from workshop import annual_global_anomaly, open_tas

DATA = Path('data/demo/tas_demo.nc')
if not DATA.exists():
    raise FileNotFoundError('Run: make demo-data')

xr.open_dataset(DATA)

## Python/xarray solution

The Python implementation exposes each conceptual step and is easy to customize.

In [ ]:
tas = open_tas(DATA)
weights = np.cos(np.deg2rad(tas.lat))
global_monthly = tas.weighted(weights).mean(('lat', 'lon'))
global_annual = global_monthly.groupby('time.year').mean('time')
baseline = global_annual.sel(year=slice(1981, 2010)).mean('year')
python_anomaly = global_annual - baseline
python_anomaly.name = 'tas_anom'
python_anomaly

## CDO/NCO solution

The same standard preprocessing can be expressed as a short operator chain. CDO evaluates chained operators from right to left. NCO then gives the output a purpose-specific variable name and attributes.

In [ ]:
%%bash
set -euo pipefail
mkdir -p outputs/opening/cdo

cdo -L -f nc4c -z zip_4 -yearmean -fldmean -subc,273.15 \
  data/demo/tas_demo.nc outputs/opening/cdo/annual_global.nc

baseline=$(cdo -s output -timmean -selyear,1981/2010 \
  outputs/opening/cdo/annual_global.nc | xargs)

cdo -L -f nc4c -z zip_4 subc,"${baseline}" \
  outputs/opening/cdo/annual_global.nc outputs/opening/cdo/anomaly.nc

ncrename -O -v tas,tas_anom outputs/opening/cdo/anomaly.nc
ncatted -O -a anomaly_reference_period,tas_anom,o,c,'1981-2010' \
  outputs/opening/cdo/anomaly.nc

echo "Baseline: ${baseline} degC"
cdo -s infon outputs/opening/cdo/anomaly.nc | sed -n '1,5p'

## Numerical agreement

Scientific equivalence means agreement within an explicit tolerance, not identical file bytes.

In [ ]:
cdo_anomaly = xr.open_dataset('outputs/opening/cdo/anomaly.nc', use_cftime=True)['tas_anom'].squeeze()
difference = np.max(np.abs(cdo_anomaly.values - python_anomaly.values))
print(f'Maximum absolute difference: {difference:.3e} degC')
assert difference < 2.0e-4
print('PASS: CDO and xarray agree within tolerance.')

## Python visualization of the CDO-prepared result

Python now receives a tiny, analysis-ready one-dimensional series rather than the full monthly three-dimensional field.

In [ ]:
!python scripts/make_stripes.py outputs/opening/cdo/anomaly.nc outputs/opening/climate_stripes.png

from IPython.display import Image, display
display(Image('outputs/opening/climate_stripes.png'))

## Takeaway

- CDO/NCO is concise for routine, standardized preprocessing.
- Python is expressive for validation, visualization, interpretation, and custom logic.
- The strongest workflow uses both where they fit best.